# 04 — Final Model: Blend (XGB + Ridge) + SHAP + Permutation Importance
XGB(0.55) + Ridge(0.45) blended model with linear calibration,  
SHAP analysis, and Permutation Importance under LOGO validation.

**Run after:** `01_eda_correlation.ipynb`  
Outputs → `artifacts/`, `model/`


In [ ]:
import warnings, os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor
import shap

warnings.filterwarnings("ignore")
np.random.seed(42); random.seed(42)
sns.set_style("whitegrid")
os.makedirs("artifacts", exist_ok=True)
os.makedirs("model", exist_ok=True)

TARGETS  = ["Leaf_TPC","Root_TPC","Leaf_TFC","Root_TFC"]
FEATURES = [
    "Temp","Humid","CO2ppm","TChl","Car",
    "Dio-RC","Eto-RC","Fv-Fm",
    "Leaf_ExtractionYield","Root_ExtractionYield",
    "month_sin","month_cos"
]
print("Ready.")


## 1. Load Data

In [2]:
def scenario_label(co2):
    if co2 < 633:   return "SSP1-2.6"
    elif co2 < 961: return "SSP3-7.0"
    else:           return "SSP5-8.5"

def ensure_month_feats(df):
    if "month_sin" not in df.columns and "month" in df.columns:
        m = df["month"].astype(float)
        df["month_sin"] = np.sin(2*np.pi*m/12)
        df["month_cos"] = np.cos(2*np.pi*m/12)
    return df

PROC_P = "data/processed/processed.csv"
if os.path.exists(PROC_P):
    df = pd.read_csv(PROC_P)
else:
    df = pd.read_csv("rawdata.csv")
    df = ensure_month_feats(df)
    if "scenario_group" not in df.columns:
        df["scenario_group"] = df["CO2ppm"].apply(scenario_label)
    num_feats = [f for f in FEATURES if f not in ["month_sin","month_cos"]]
    df[num_feats] = RobustScaler().fit_transform(df[num_feats])

df = ensure_month_feats(df)
if "scenario_group" not in df.columns:
    df["scenario_group"] = df["CO2ppm"].apply(scenario_label)

X      = df[FEATURES].values
X_df   = df[FEATURES].copy()
y      = df[TARGETS].values
groups = df["scenario_group"].values
print(f"X: {X.shape}, y: {y.shape}")

X: (405, 12), y: (405, 4)


## 2. Blending: XGB(0.55) + Ridge(0.45)

In [3]:
# ── 원본 제출코드에서 사용한 최적 가중치 그대로 ──
BLEND_W = 0.55   # XGB 비중

def xgb_predict_single(X_tr, y_tr_k, X_te):
    m = XGBRegressor(
        n_estimators=500, learning_rate=0.08, max_depth=5,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        objective="reg:squarederror", random_state=42, verbosity=0
    )
    m.fit(X_tr, y_tr_k)
    return m.predict(X_te)

def blend_predict(X_tr, y_tr, X_te):
    """타깃별 XGB + Ridge 예측 블렌딩"""
    preds = np.zeros((len(X_te), len(TARGETS)))
    for k in range(len(TARGETS)):
        xgb_pred   = xgb_predict_single(X_tr, y_tr[:,k], X_te)
        ridge_pred = Ridge(alpha=1.0).fit(X_tr, y_tr[:,k]).predict(X_te)
        preds[:,k]  = BLEND_W * xgb_pred + (1 - BLEND_W) * ridge_pred
    return preds

# LOGO 평가
logo  = LeaveOneGroupOut()
preds_blend = np.zeros_like(y, dtype=float)
for tr, te in logo.split(X, y, groups):
    preds_blend[te] = blend_predict(X[tr], y[tr], X[te])

def metrics_table(y_true, y_pred):
    rows = []
    for i, tgt in enumerate(TARGETS):
        rows.append({
            "target": tgt,
            "MAE":  mean_absolute_error(y_true[:,i], y_pred[:,i]),
            "RMSE": np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])),
            "R2":   r2_score(y_true[:,i], y_pred[:,i])
        })
    df_m = pd.DataFrame(rows)
    df_m.loc["mean"] = ["MEAN", df_m["MAE"].mean(), df_m["RMSE"].mean(), df_m["R2"].mean()]
    return df_m

mt_blend = metrics_table(y, preds_blend)
print("=== Blend LOGO Performance ===")
print(mt_blend.round(4).to_string(index=False))
mt_blend.to_csv("artifacts/logo_blend_final.csv", index=False)

=== Blend LOGO Performance ===
  target    MAE   RMSE      R2
Leaf_TPC 0.4279 0.5579 -0.2158
Root_TPC 0.3748 0.5011  0.1666
Leaf_TFC 0.7106 1.0578  0.8309
Root_TFC 0.0902 0.1148  0.2890
    MEAN 0.4009 0.5579  0.2677


## 3. Scenario-wise LOGO Performance

In [4]:
rows_scen = []
for tr, te in logo.split(X, y, groups):
    scen = np.unique(groups[te])[0]
    pred = blend_predict(X[tr], y[tr], X[te])
    rows_scen.append({
        "scenario": scen,
        "MAE":  float(np.mean([mean_absolute_error(y[te][:,i], pred[:,i]) for i in range(4)])),
        "RMSE": float(np.mean([np.sqrt(mean_squared_error(y[te][:,i], pred[:,i])) for i in range(4)])),
        "R2":   float(np.mean([r2_score(y[te][:,i], pred[:,i]) for i in range(4)])),
    })

df_scen = pd.DataFrame(rows_scen)
print("=== Scenario-wise LOGO ===")
print(df_scen.round(4).to_string(index=False))
df_scen.to_csv("artifacts/logo_blend_by_scenario.csv", index=False)

=== Scenario-wise LOGO ===
scenario    MAE   RMSE      R2
SSP1-2.6 0.4187 0.5326  0.3025
SSP3-7.0 0.4644 0.6502  0.2564
SSP5-8.5 0.3227 0.4449 -0.7116


## 4. SHAP Analysis (XGBoost, full-data fit 기준)

> **Note:** SHAP은 전체 데이터(405행)로 학습한 XGB 모델의 in-sample 설명입니다.  
> LOGO 교차검증 모델(unseen scenario에 대한 예측)과는 별개이며,  
> "어떤 피처가 모델의 예측에 영향을 주는가"를 이해하기 위한 보조 분석입니다.


In [ ]:
# SHAP: XGB 단일 모델 기준 (Ridge는 선형 계수로 충분하므로 XGB 기반 해석)
# 전체 405행으로 학습한 in-sample 모델 → in-sample SHAP explanation
# (LOGO fold별 SHAP 집계는 계산 비용이 매우 높아 여기서는 full-data fit 사용)

xgb_full_models = []
for k in range(len(TARGETS)):
    m = XGBRegressor(
        n_estimators=500, learning_rate=0.08, max_depth=5,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        objective="reg:squarederror", random_state=42, verbosity=0
    )
    m.fit(X, y[:,k])
    xgb_full_models.append(m)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for k, tgt in enumerate(TARGETS):
    explainer   = shap.TreeExplainer(xgb_full_models[k])
    shap_values = explainer.shap_values(X)
    plt.sca(axes[k])
    shap.summary_plot(shap_values, X_df, show=False, max_display=12)
    axes[k].set_title(f"SHAP — {tgt} (in-sample, full-data fit)")

plt.suptitle("SHAP Beeswarm: feature contribution to prediction\n"
             "(in-sample explanation — not LOGO)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("artifacts/shap_beeswarm_all_targets.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → artifacts/shap_beeswarm_all_targets.png")


## 5. Permutation Importance (in-sample, XGB full-data fit 기준)

> **Note:** 아래 PI는 전체 데이터로 학습한 XGB 모델에 대한 in-sample Permutation Importance입니다.  
> 진짜 LOGO 기반 PI(각 fold에서 재학습 후 test set PI 집계)와는 다릅니다.  
> 그러나 "어떤 피처가 예측에 중요한가"의 상대적 순위를 파악하는 데 유효합니다.


In [ ]:
# Permutation Importance — xgb_full_models (full-data fit) 기준 in-sample PI
# 각 피처를 무작위로 섞었을 때 MSE가 얼마나 증가하는지 측정 (n_repeats=10)
# MDI(RF feature_importances_)보다 신뢰할 수 있지만,
# 여기서는 in-sample이므로 unseen scenario에서의 중요도와 다를 수 있음

class XGBWrapSingle(RegressorMixin, BaseEstimator):
    """permutation_importance 호환을 위한 단일 타깃 래퍼"""
    def __init__(self, k=0): self.k = k
    def fit(self, X, y):     return self
    def predict(self, X):    return xgb_full_models[self.k].predict(X)

pi_results = {}
for k, tgt in enumerate(TARGETS):
    r = permutation_importance(
        XGBWrapSingle(k=k), X, y[:,k],
        n_repeats=10, random_state=42,
        scoring="neg_mean_squared_error", n_jobs=-1
    )
    pi_results[tgt] = pd.Series(r.importances_mean, index=FEATURES)

pi_df   = pd.DataFrame(pi_results)
pi_norm = pi_df.div(pi_df.sum(axis=0).replace(0, np.nan), axis=1)

plt.figure(figsize=(9, 5))
sns.heatmap(pi_norm.T, annot=True, fmt=".2f", cmap="viridis",
            linewidths=0.3, cbar_kws={"label": "Normalized PI"})
plt.title("Permutation Importance (normalized per target)\nin-sample, XGB full-data fit")
plt.xlabel("Feature"); plt.ylabel("Target")
plt.tight_layout()
plt.savefig("artifacts/permutation_importance_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
pi_df.to_csv("artifacts/permutation_importance_raw.csv")
print("Saved → artifacts/permutation_importance_heatmap.png")


## 6. Save Final Model & Metadata

In [7]:
# XGB 모델 저장 (타깃별)
for k, tgt in enumerate(TARGETS):
    xgb_full_models[k].save_model(f"model/xgb_{tgt}.json")

# 메타데이터 저장
meta = {
    "model": "Blend XGB(0.55) + Ridge(0.45)",
    "blend_weight_xgb": BLEND_W,
    "features": FEATURES,
    "targets": TARGETS,
    "logo_mean_r2": float(mt_blend.loc[mt_blend["target"]=="MEAN","R2"].values[0]),
    "logo_mean_mae": float(mt_blend.loc[mt_blend["target"]=="MEAN","MAE"].values[0]),
}
with open("model/metadata.json", "w") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Saved → model/metadata.json")
print("Saved → model/xgb_*.json")

Saved → model/metadata.json
Saved → model/xgb_*.json
